<a href="https://colab.research.google.com/github/N17-R4M/bigdata-lab-FFLM/blob/main/Analise_Explorat%C3%B3ria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto de Big Data: Análise do Exame Nacional (2024)

Este notebook apresenta a etapa de Análise Exploratória de Dados (EDA) e Limpeza das bases do exame.
Para garantir a colaboração e evitar limitações de memória, os dados estão hospedados no Google Drive, permitindo que toda a nova equipe acesse a mesma fonte de dados de forma sincronizada.

### 🚩 Nota para a Equipe (Setup do Ambiente)

Para que este notebook funcione perfeitamente na máquina de qualquer membro do grupo, sigam estes passos:
1. Abram o e-mail neutro do projeto e acessem o Google Drive compartilhado.
2. Garantam que os arquivos `PARTICIPANTES_2024.csv`, `RESULTADOS_2024.csv` e `ITENS_PROVA_2024.csv` estão salvos dentro de uma pasta chamada `Projeto_ENEM` no Drive.
3. Ao executar a primeira célula deste código, o Colab pedirá permissão para acessar o Drive. Basta autorizar e rodar o resto do notebook!

In [ ]:
# Importando as bibliotecas de manipulação e visualização
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração de estilo para os gráficos
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option('display.max_columns', None)

# 1. Conectando ao Drive da equipe
from google.colab import drive
print("Conectando ao Google Drive...")
drive.mount('/content/drive')

# 2. Apontando para a pasta correta
# ATENÇÃO: Se a pasta no Drive tiver outro nome, mude aqui:
caminho_drive = '/content/drive/MyDrive/Projeto_ENEM/'

# 3. Carregando as bases de dados
print("\nCarregando as bases de dados (Isso pode levar alguns instantes)...")
df_participantes = pd.read_csv(caminho_drive + 'PARTICIPANTES_2024.csv', encoding='latin1')
df_resultados = pd.read_csv(caminho_drive + 'RESULTADOS_2024.csv', encoding='latin1')
df_itens = pd.read_csv(caminho_drive + 'ITENS_PROVA_2024.csv', encoding='latin1')

print("✅ Bases carregadas com sucesso direto do Google Drive!")

##Análise Exploratória

In [ ]:
# Criando uma função para não repetir código e manter o notebook organizado
def analise_exploratoria_inicial(df, nome_base):
    print(f"\n{'='*50}")
    print(f"🕵️‍♂️ ANÁLISE EXPLORATÓRIA: {nome_base.upper()}")
    print(f"{'='*50}")

    # 1. Volume de Dados
    linhas, colunas = df.shape
    print(f"📌 Volume: {linhas:,} registros (linhas) e {colunas} atributos (colunas).")

    # 2. Dados Faltantes (Visão Geral)
    total_nulos = df.isnull().sum().sum()
    print(f"📌 Total de valores em branco na base toda: {total_nulos:,}")

    # 3. Amostra dos Dados
    print(f"\nVisualização das primeiras 3 linhas:")
    display(df.head(3))

# Executando a análise para as 3 bases
analise_exploratoria_inicial(df_participantes, "Participantes")
analise_exploratoria_inicial(df_resultados, "Resultados")
analise_exploratoria_inicial(df_itens, "Itens da Prova")

##A Limpeza Profunda

In [ ]:
print("Iniciando a rotina de Limpeza e Estruturação de Dados...\n")

# 1. LIMPEZA DA BASE DE PARTICIPANTES
colunas_participantes = ['NU_INSCRICAO', 'TP_SEXO', 'TP_COR_RACA', 'NU_IDADE', 'SG_UF_PROVA']
colunas_existentes = [col for col in colunas_participantes if col in df_participantes.columns]
df_part_limpo = df_participantes[colunas_existentes].copy()
df_part_limpo = df_part_limpo.dropna(subset=['NU_IDADE', 'SG_UF_PROVA'])
print(f"✅ Participantes limpos: Retivemos {len(df_part_limpo):,} registros válidos.")

# 2. LIMPEZA DA BASE DE RESULTADOS
colunas_resultados = ['NU_INSCRICAO', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']
colunas_existentes_res = [col for col in colunas_resultados if col in df_resultados.columns]
df_res_limpo = df_resultados[colunas_existentes_res].copy()

if 'NU_NOTA_REDACAO' in df_res_limpo.columns:
    df_res_limpo = df_res_limpo.dropna(subset=['NU_NOTA_REDACAO'])
df_res_limpo = df_res_limpo.fillna(0)
print(f"✅ Resultados limpos: Retivemos {len(df_res_limpo):,} registros válidos.")

# 3. LIMPEZA DA BASE DE ITENS
df_itens_limpo = df_itens.drop_duplicates()
print(f"✅ Itens da Prova limpos: Retivemos {len(df_itens_limpo):,} itens únicos.")

##Unindo os Dados e Gerando Insights Visuais

In [ ]:
# Unindo (Merge) Participantes e Resultados usando o Número de Inscrição como chave
# Usamos 'inner' para manter apenas quem tem dados sociodemográficos E notas
if 'NU_INSCRICAO' in df_part_limpo.columns and 'NU_INSCRICAO' in df_res_limpo.columns:
    df_final = pd.merge(df_part_limpo, df_res_limpo, on='NU_INSCRICAO', how='inner')
    print(f"\n🔗 Bases de Participantes e Resultados cruzadas com sucesso!")
    print(f"Base final para análise contém {len(df_final):,} registros.")

    # ---------------------------------------------------------
    # Gráfico 1: A distribuição das Notas de Matemática
    # ---------------------------------------------------------
    if 'NU_NOTA_MT' in df_final.columns:
        plt.figure(figsize=(10, 5))
        sns.histplot(df_final['NU_NOTA_MT'], bins=30, kde=True, color='blue')
        plt.title('Distribuição do Desempenho em Matemática', fontsize=14, fontweight='bold')
        plt.xlabel('Nota de Matemática', fontsize=12)
        plt.ylabel('Frequência de Alunos', fontsize=12)
        plt.show()

    # ---------------------------------------------------------
    # Gráfico 2: Participantes por Estado (As 10 maiores concentrações)
    # ---------------------------------------------------------
    if 'SG_UF_PROVA' in df_final.columns:
        plt.figure(figsize=(10, 5))
        top_10_estados = df_final['SG_UF_PROVA'].value_counts().head(10)
        sns.barplot(x=top_10_estados.index, y=top_10_estados.values, palette='viridis')
        plt.title('Top 10 Estados com Maior Número de Participantes', fontsize=14, fontweight='bold')
        plt.xlabel('Estado (UF)', fontsize=12)
        plt.ylabel('Número de Participantes Válidos', fontsize=12)
        plt.show()
else:
    print("\nA coluna 'NU_INSCRICAO' não foi encontrada para fazer o cruzamento. Verifique o nome das colunas.")